In [1]:
from dotenv import load_dotenv
import os
import finnhub
import pandas as pd
import numpy as np

from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame
from alpaca.trading.client import TradingClient
import plotly.express as px
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error 



In [2]:
load_dotenv()

KEY = os.getenv("key")
SECRET = os.getenv("secret")

In [3]:
client = StockHistoricalDataClient(KEY, SECRET)

request = StockBarsRequest(
    symbol_or_symbols = "NVDA",
    timeframe = TimeFrame.Day,
    start = "2026-01-01",
)
bars = client.get_stock_bars(request)

df = bars.df

df

open      high     low   close  \
symbol timestamp                                                      
NVDA   2026-01-02 05:00:00+00:00  189.835  192.9299  188.26  188.85   
       2026-01-05 05:00:00+00:00  191.760  193.6300  186.15  188.12   
       2026-01-06 05:00:00+00:00  190.520  192.1740  186.82  187.24   
       2026-01-07 05:00:00+00:00  188.570  191.3700  186.56  189.11   
       2026-01-08 05:00:00+00:00  189.110  189.5500  183.71  185.04   
...                                   ...       ...     ...     ...   
       2026-09-08 04:00:00+00:00  233.110  233.7100  224.85  225.73   
       2026-09-09 04:00:00+00:00  225.275  226.1800  223.46  223.67   
       2026-09-10 04:00:00+00:00  220.525  220.9900  217.20  218.36   
       2026-09-11 04:00:00+00:00  221.235  222.0000  218.15  218.29   
       2026-09-14 04:00:00+00:00  211.240  212.3900  208.93  212.20   

                                       volume  trade_count        vwap  
symbol timestamp                                                        
NVDA   2026-01-02 05:00:00+00:00  148240462.0    2260604.0  189.912199  
       2026-01-05 05:00:00+00:00  182965768.0    2574019.0  188.757392  
       2026-01-06 05:00:00+00:00  176862604.0    2519447.0  189.058048  
       2026-01-07 05:00:00+00:00  153543174.0    2292255.0  189.527040  
       2026-01-08 05:00:00+00:00  172457048.0    2521746.0  185.358816  
...                                       ...          ...         ...  
       2026-09-08 04:00:00+00:00  123863958.0    2405736.0  227.183114  
       2026-09-09 04:00:00+00:00   83765394.0    1827357.0  224.230606  
       2026-09-10 04:00:00+00:00  106720602.0    2202079.0  218.423135  
       2026-09-11 04:00:00+00:00   90036579.0    2013738.0  219.556789  
       2026-09-14 04:00:00+00:00   77067681.0    1987583.0  210.863471  

[175 rows x 7 columns]

In [4]:
#Setter opp klienten

finnhub_client = finnhub.Client(api_key=os.getenv("finnhub_key"))

#Henter alle grunnleggende data for NVIDIA
data = finnhub_client.company_basic_financials("NVDA", "all")
quarterly_series = data["series"]["quarterly"]

print(data)
#Dette gir meg en stor dictonary med masse metrics under data["metric"]. f.eks peNormalizedAnnual, roeTTM, netProfitMarginTTM osv.

{'metric': {'10DayAverageTradingVolume': 125.74631, '13WeekPriceReturnDaily': 8.9163, '26WeekPriceReturnDaily': 19.5127, '3MonthADReturnStd': 40.5757, '3MonthAverageTradingVolume': 144.10357, '52WeekHigh': 236.54, '52WeekHighDate': '2026-05-14', '52WeekLow': 164.27, '52WeekLowDate': '2026-03-30', '52WeekPriceReturnDaily': 23.2093, '5DayPriceReturnDaily': -5.2396, 'assetTurnoverAnnual': 1.0442, 'assetTurnoverTTM': 1.2788, 'beta': 2.2238405, 'bookValuePerShareAnnual': 6.4719, 'bookValuePerShareQuarterly': 9.4829, 'bookValueShareGrowth5Y': 56.87, 'capexCagr5Y': 39.89, 'cashFlowPerShareAnnual': 3.9778, 'cashFlowPerShareQuarterly': 5.2597, 'cashFlowPerShareTTM': 8.20683, 'cashPerSharePerShareAnnual': 2.5739, 'cashPerSharePerShareQuarterly': 4.1152, 'currentDividendYieldTTM': 0.1288, 'currentEv/freeCashFlowAnnual': 54.5297, 'currentEv/freeCashFlowTTM': 41.5076, 'currentRatioAnnual': 3.9053, 'currentRatioQuarterly': 4.5889, 'dividendGrowthRate5Y': 20.05, 'dividendIndicatedAnnual': 1, 'dividen

In [5]:
#Selskaps profil
profile = finnhub_client.company_profile2(symbol="NVDA")
print(profile)

#Dette gir meg ting som markedsverdi, sektor, land, børs, antall utestående aksjer, short interest osv. Dette er ting som kan være nyttig for å lage en regresjonsmodell.

{'ticker': 'NVDA', 'name': 'NVIDIA Corp', 'country': 'US', 'currency': 'USD', 'estimateCurrency': 'USD', 'exchange': 'NASDAQ NMS - GLOBAL MARKET', 'ipo': '1999-01-22', 'marketCapitalization': 5260789.014704283, 'logo': 'https://static2.finnhub.io/file/publicdatany/finnhubimage/stock_logo/942959144056.png', 'shareOutstanding': 24200, 'finnhubIndustry': 'Semiconductors', 'phone': '13026365401', 'weburl': 'https://www.nvidia.com/', 'floatingShare': 23228.1}


In [6]:
#Gjør det til en pandas dataframe for enklere dataanalyse

metrics = data["metric"]
df.fundamentals = pd.DataFrame(list(metrics.items()), columns=["Nøkkeltall", "Verdi"])
df.fundamentals

C:\Users\lukat\AppData\Local\Temp\ipykernel_18148\2103141244.py:4: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  df.fundamentals = pd.DataFrame(list(metrics.items()), columns=["Nøkkeltall", "Verdi"])


,Nøkkeltall,Verdi
0,10DayAverageTradingVolume,125.74631
1,13WeekPriceReturnDaily,8.9163
2,26WeekPriceReturnDaily,19.5127
3,3MonthADReturnStd,40.5757
4,3MonthAverageTradingVolume,144.10357
...,...,...
128,tangibleBookValuePerShareQuarterly,9.3588
129,tbvCagr5Y,61.18
130,totalDebt/totalEquityAnnual,0.0538
131,totalDebt/totalEquityQuarterly,0.1457


In [7]:
#Velge nøkkeltall man trenger selv (Hva som kan være nyttig for å lage en regresjonsmodell for NVIDIA)
fundamentals_metrics = ["peNormalizedAnnual", "roeTTM", "netProfitMarginTTM",  "currentRatioAnnual",  "grossMarginTTM"]
for n in fundamentals_metrics:
    print(n, ":", metrics.get(n))

peNormalizedAnnual : 43.8154
roeTTM : 110.11
netProfitMarginTTM : 63.66
currentRatioAnnual : 3.9053
grossMarginTTM : 74.67


In [8]:
#Dette er en kombinert datasett for en samlet tabell bare for finnhub verdiene
fundamental_metrics = ["peTTM", "roeTTM", "netMargin", "currentRatio", "grossMargin"]

fundamentals = {}

for metric in fundamental_metrics:
    entries = quarterly_series.get(metric, [])
    for entry in entries:
        date = entry["period"]
        value = entry["v"]
        fundamentals.setdefault(date, {})[metric] = value

df_fundamentals = pd.DataFrame.from_dict(fundamentals, orient="index")
df_fundamentals.index = pd.to_datetime(df_fundamentals.index)
df_fundamentals = df_fundamentals.sort_index()

df_fundamentals

,peTTM,roeTTM,netMargin,currentRatio,grossMargin
1997-03-30,NaN,NaN,-18.0923,NaN,-2.2000
1997-06-29,NaN,NaN,-210.8333,NaN,-24.0000
1997-09-28,NaN,NaN,-0.4705,NaN,0.1679
1997-12-31,NaN,NaN,0.0605,NaN,0.3058
1998-04-26,NaN,NaN,-0.0361,NaN,0.2615
...,...,...,...,...,...
2025-07-27,49.8020,1.0522,0.5653,4.2140,0.7242
2025-10-26,46.9083,1.0382,0.5598,4.4676,0.7341
2026-01-25,37.7391,1.0437,0.6306,3.9053,0.7500
2026-04-26,32.9774,1.1166,0.7146,3.4408,0.7493


TABELLEN KOMBINERTE DATASETT UNDER


In [9]:
# Gjør klar df (markedsdata) for merge
df_market = df.reset_index()
df_market['timestamp'] = pd.to_datetime(df_market['timestamp']).dt.tz_localize(None)

# Gjør klar df_fundamentals for merge
df_fund = df_fundamentals.reset_index().rename(columns={'index': 'timestamp'})
df_fund['timestamp'] = pd.to_datetime(df_fund['timestamp'])

# Sorter begge på timestamp (påkrevd for merge_asof)
df_market = df_market.sort_values('timestamp')
df_fund = df_fund.sort_values('timestamp')

# Koble sammen: for hver handelsdag, hent siste kjente fundamentalverdi
df_combined = pd.merge_asof(df_market, df_fund, on='timestamp', direction='backward')

df_combined

,symbol,timestamp,open,high,low,close,volume,trade_count,vwap,peTTM,roeTTM,netMargin,currentRatio,grossMargin
0,NVDA,2026-01-02 05:00:00,189.835,192.9299,188.26,188.85,148240462.0,2260604.0,189.912199,46.9083,1.0382,0.5598,4.4676,0.7341
1,NVDA,2026-01-05 05:00:00,191.760,193.6300,186.15,188.12,182965768.0,2574019.0,188.757392,46.9083,1.0382,0.5598,4.4676,0.7341
2,NVDA,2026-01-06 05:00:00,190.520,192.1740,186.82,187.24,176862604.0,2519447.0,189.058048,46.9083,1.0382,0.5598,4.4676,0.7341
3,NVDA,2026-01-07 05:00:00,188.570,191.3700,186.56,189.11,153543174.0,2292255.0,189.527040,46.9083,1.0382,0.5598,4.4676,0.7341
4,NVDA,2026-01-08 05:00:00,189.110,189.5500,183.71,185.04,172457048.0,2521746.0,185.358816,46.9083,1.0382,0.5598,4.4676,0.7341
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
170,NVDA,2026-09-08 04:00:00,233.110,233.7100,224.85,225.73,123863958.0,2405736.0,227.183114,24.6556,1.1011,0.6203,4.5889,0.7498
171,NVDA,2026-09-09 04:00:00,225.275,226.1800,223.46,223.67,83765394.0,1827357.0,224.230606,24.6556,1.1011,0.6203,4.5889,0.7498
172,NVDA,2026-09-10 04:00:00,220.525,220.9900,217.20,218.36,106720602.0,2202079.0,218.423135,24.6556,1.1011,0.6203,4.5889,0.7498
173,NVDA,2026-09-11 04:00:00,221.235,222.0000,218.15,218.29,90036579.0,2013738.0,219.556789,24.6556,1.1011,0.6203,4.5889,0.7498


In [10]:
delta = df_combined["close"].diff()

gain = delta.where(delta > 0, 0)
loss = -delta.where(delta < 0, 0)

avg_gain = gain.rolling(window=14).mean()
avg_loss = loss.rolling(window=14).mean()

rs = avg_gain / avg_loss

df_combined["rsi14"] = 100 - (100 / (1 + rs))

px.line(
    df_combined.reset_index().dropna(),
    x="timestamp",
    y="rsi14",
    title="NVDA RSI14"
).show()


In [11]:
sma20 = df_combined["close"].rolling(window=20).mean()
std20 = df_combined["close"].rolling(window=20).std()

df_combined["upper_band"] = sma20 + (std20 * 2)
df_combined["lower_band"] = sma20 - (std20 * 2)

px.line(
    df_combined.reset_index().dropna(),
    x="timestamp",
    y=["close", "upper_band", "lower_band"],
    title="NVDA Bollinger Bands"
).show()

In [12]:
high_low = df_combined["high"] - df_combined["low"]
high_close = abs(df_combined["high"] - df_combined["close"].shift(1))
low_close = abs(df_combined["low"] - df_combined["close"].shift(1))

tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)

df_combined["atr14"] = tr.rolling(window=14).mean()

px.line(
    df_combined.reset_index().dropna(),
    x="timestamp",
    y= "atr14",
    title="NVDA ATR14"
).show()

In [13]:
#Gjennomsnittprisen over en bestemt periode

df_combined["sma50"] = df_combined["close"].rolling(window=50).mean()
df_combined["sma100"] = df_combined["close"].rolling(window=100).mean()

px.line(
    df_combined.reset_index(),
    x="timestamp",
    y=["close", "sma50", "sma100"],
    title="NVDA Close Price with SMA50 and SMA100"
).show()

In [14]:
df_combined.isna().sum()



symbol           0
timestamp        0
open             0
high             0
low              0
close            0
volume           0
trade_count      0
vwap             0
peTTM            0
roeTTM           0
netMargin        0
currentRatio     0
grossMargin      0
rsi14           13
upper_band      19
lower_band      19
atr14           13
sma50           49
sma100          99
dtype: int64

In [15]:
df_combined = df_combined.dropna()

In [16]:
df_combined.describe()

,timestamp,open,high,low,close,volume,trade_count,vwap,peTTM,roeTTM,netMargin,currentRatio,grossMargin,rsi14,upper_band,lower_band,atr14,sma50,sma100
count,76,76.000000,76.000000,76.000000,76.000000,7.600000e+01,7.600000e+01,76.000000,76.000000,76.000000,76.000000,76.000000,76.000000,76.000000,76.000000,76.000000,76.000000,76.000000,76.000000
mean,2026-07-20 18:31:34.736842,211.229875,214.146253,207.724522,210.739342,1.400003e+08,2.450962e+06,210.837832,29.144992,1.109462,0.671172,3.969530,0.749530,50.295105,225.461181,195.598569,7.400346,207.648176,200.426422
min,2026-05-27 04:00:00,192.370000,195.550000,189.800000,190.010000,7.650232e+07,1.741900e+06,192.174850,24.655600,1.101100,0.620300,3.440800,0.749300,32.673118,212.827655,188.297090,5.438407,198.087600,191.741300
25%,2026-06-23 22:00:00,203.982500,207.182500,199.490000,203.467500,1.136204e+08,2.160200e+06,203.070047,24.655600,1.101100,0.620300,3.440800,0.749300,42.001052,216.417926,189.758867,6.801289,206.265850,196.104775
50%,2026-07-21 16:00:00,211.210000,213.330000,207.270000,211.050000,1.351393e+08,2.394993e+06,210.221473,32.977400,1.116600,0.714600,3.440800,0.749300,49.700314,226.659831,191.590305,7.588636,208.314100,199.028050
75%,2026-08-17 10:00:00,220.468750,222.205000,216.490000,218.307500,1.545439e+08,2.607926e+06,219.340525,32.977400,1.116600,0.714600,4.588900,0.749800,57.458130,232.465868,200.056251,7.874802,209.751850,204.885250
max,2026-09-14 04:00:00,233.110000,234.760000,229.630000,230.360000,2.996411e+08,4.707580e+06,231.817409,32.977400,1.116600,0.714600,4.588900,0.749800,77.411027,235.915069,208.608221,8.757064,212.931400,211.363700
std,NaN,10.447244,10.197314,10.439210,10.007226,4.121744e+07,4.477871e+05,10.136401,4.175474,0.007777,0.047315,0.576061,0.000251,10.518190,7.922884,7.037516,0.786132,3.041152,5.699839


In [17]:
corr = df_combined.corr(numeric_only=True)

px.imshow(
    corr,
    zmin=-1,
    zmax=1,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    title = "Correlation Matrix"
).show()

In [18]:
df_combined = df_combined.reset_index()

px.line(
    df_combined,
    x="timestamp",
    y="close",
    title="NVDA Closing Prices",
).show()

In [19]:
returns = df_combined["close"].pct_change().dropna()
px.histogram(
    returns,
    nbins=20,
    title="NVDA Daily Returns Distribution",).show()

In [20]:
px.box(
    returns,
    title="NVDA Daily Returns Box Plot"
).show()



In [21]:
go.Figure(go.Candlestick(
    x=df_combined["timestamp"],
    open=df_combined["open"],
    high=df_combined["high"],
    low=df_combined["low"],
    close=df_combined["close"],
)).show()

In [22]:
df_combined["target"] = df_combined["close"].shift(-1) / df_combined["close"] - 1

# Fjern siste rad, som mangler target
df_combined = df_combined.dropna(subset=["target"]) #fjerner siste rad som inneholder NaN i target-kolonnen

df_combined[["timestamp", "close", "target"]].tail(10)

,timestamp,close,target
65,2026-08-28 04:00:00,217.55,0.014847
66,2026-08-31 04:00:00,220.78,-0.015128
67,2026-09-01 04:00:00,217.44,0.032055
68,2026-09-02 04:00:00,224.41,0.018003
69,2026-09-03 04:00:00,228.45,0.008361
70,2026-09-04 04:00:00,230.36,-0.020099
71,2026-09-08 04:00:00,225.73,-0.009126
72,2026-09-09 04:00:00,223.67,-0.023740
73,2026-09-10 04:00:00,218.36,-0.000321
74,2026-09-11 04:00:00,218.29,-0.027899


Del datasettet i test- og treningsdata basert på tidsrekkefølge

In [23]:
split = int(len(df_combined) * 0.8) # finner hvor 80% grensen er

train_df = df_combined.iloc[:split] # De første ELDSTE 80% av dagene
test_df = df_combined.iloc[split:]  # De siste nyeste 20% av dagene

X_train = train_df.drop(columns=['target', "symbol", "timestamp"])      #features for trening
y_train = train_df['target']                      #fasit for trening

X_test = test_df.drop(columns=['target', "symbol", "timestamp"])         #features for test
y_test = test_df['target']                        #fasit for test



Tren en regresjonsmodell, for eksempel lineær regresjon.
o Eller SVR for en smule mer utfordring


In [24]:
model = LinearRegression() #oppretter modellen
model.fit(X_train, y_train) # "lærer" mønsteret på treningsdataene

y_pred = model.predict(X_test) #bruker det lærte mønsteret til å predikere på testdataene


In [ ]:
Evaluer modellen ved hjelp av mål som MAE, RMSE og R².

In [25]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)  # Calculate RMSE from MSE
r2 = r2_score(y_test, y_pred)

print(f"Mål: MAE, Verdi: {mae}")
print(f"Mål: RMSE, Verdi: {rmse}")
print(f"Mål: R², Verdi: {r2}")

Mål: MAE, Verdi: 0.024445447674195705
Mål: RMSE, Verdi: 0.03148830650492399
Mål: R², Verdi: 0.00883240266070251


In [28]:
viktig = pd.DataFrame({
    "feature": X_train.columns,
    "coefficient": model.coef_
}).sort_values(by="coefficient", key=abs, ascending=False)

viktig

,feature,coefficient
6,trade_count,-3.739618e-09
5,volume,6.752495e-11
13,rsi14,5.053251e-14
3,low,3.702797e-14
4,close,2.973271e-14
7,vwap,2.970410e-14
0,index,2.382129e-14
1,open,1.918767e-14
2,high,1.766271e-14
15,lower_band,-1.590573e-14


In [31]:
# Lag en dataframe med faktisk og predikert avkastning for testperioden
pred_df = pd.DataFrame({
    "timestamp": test_df["timestamp"].values,
    "faktisk_avkastning": y_test.values,
    "predikert_avkastning": y_pred
})

# Linjegraf: faktisk vs. predikert avkastning over tid
px.line(
    pred_df,
    x="timestamp",
    y=["faktisk_avkastning", "predikert_avkastning"],
    title="NVDA: Faktisk vs. Predikert Avkastning (testdata)"
).show()